In [ ]:
import numpy as numpy
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("BDD#7_Database_Albert_School_Sephora.csv")
print(df.dtypes)

In [ ]:
df.columns.tolist()

In [ ]:
# Conversions de types
date_cols = ["transactionDate", "subscription_date", "first_purchase_dt"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["anonymized_card_code"] = df["anonymized_card_code"].astype("Int64").astype("str")


In [ ]:
# Gender : remplacer 99999 par le mode
mode_gender = df.loc[df["gender"] != 99999, "gender"].mode()[0]
df["gender"] = df["gender"].replace(99999, mode_gender)

# RFM : 99999 → NaN
df["RFM_Segment_ID"] = df["RFM_Segment_ID"].replace(99999, pd.NA)

# Age impossible → NaN
df.loc[(df["age"] <= 0) | (df["age"] > 100), "age"] = pd.NA
df["age"] = df["age"].fillna(df["age"].median())

# Flag retours
df["is_return"] = (df["salesVatEUR"] < 0).astype(int)

In [ ]:
# Flag : données 1er achat disponibles
df["has_first_purchase_data"] = df["first_purchase_dt"].notna().astype(int)

# Reconstruire age_category depuis age
bins   = [0, 15, 25, 35, 45, 60, 100]
labels = ["<15yo","15-25yo","26-35yo","36-45yo","46-60yo","60+yo"]
df["age_category"] = df["age_category"].fillna(
    pd.cut(df["age"], bins=bins, labels=labels).astype("str")
)

# Colonnes catégorielles → "inconnu"
cat_cols = ["customer_city", "Market_Desc", "brand"]
df[cat_cols] = df[cat_cols].fillna("inconnu")

# Colonnes numériques first_purchase → médiane
fp_num = ["salesVatEUR_first_purchase", "discountEUR_first_purchase", "quantity_first_purchase"]
df[fp_num] = df[fp_num].fillna(df[fp_num].median())

In [ ]:
# Harmoniser les timezones avant soustraction
df["transactionDate"] = df["transactionDate"].dt.tz_localize(None)
df["subscription_date"] = df["subscription_date"].dt.tz_localize(None)

df["days_since_sub"] = (df["transactionDate"] - df["subscription_date"]).dt.days

In [ ]:
# Features temporelles
df["day_of_week"]  = df["transactionDate"].dt.dayofweek
df["month"]        = df["transactionDate"].dt.month
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["days_since_sub"] = (df["transactionDate"] - df["subscription_date"]).dt.days

# Variable cible : jours jusqu'au prochain achat
df = df.sort_values(["anonymized_card_code", "transactionDate"])
df["next_purchase_date"] = df.groupby("anonymized_card_code")["transactionDate"].shift(-1)
df["days_to_next_purchase"] = (
    df["next_purchase_date"] - df["transactionDate"]
).dt.days

In [ ]:
df["days_since_sub"] = df["days_since_sub"].fillna(-1)  # -1 = pas d'abonnement
df["age_generation"] = df["age_generation"].fillna("inconnu")
fp_cat = ["materialCode_first_purchase", "Axe_Desc_first_purchase",
          "Market_Desc_first_purchase", "brand_first_purchase"]
df[fp_cat] = df[fp_cat].fillna("inconnu")

In [ ]:
# Résumé post-nettoyage
print("Shape :", df.shape)
print("Missing restants :\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Retours :", df["is_return"].sum(), "lignes")
print("Cible nulle (dernier achat) :", df["days_to_next_purchase"].isna().sum())

In [ ]:
print(df["days_to_next_purchase"].describe())
import matplotlib.pyplot as plt
df["days_to_next_purchase"].dropna().clip(upper=365).hist(bins=60, figsize=(12,4))
plt.title("Distribution des jours jusqu'au prochain achat (cap 365j)")
plt.xlabel("Jours")
plt.ylabel("Fréquence")
plt.show()

In [ ]:
# Est-ce que les 0 sont des lignes du même ticket ?
same_ticket = df[df["days_to_next_purchase"] == 0].groupby(
    ["anonymized_card_code", "transactionDate"]
)["anonymized_Ticket_ID"].nunique()

print("Tickets distincts dans les 0-jours :", same_ticket.value_counts().head())

# Distribution sans les 0
df[df["days_to_next_purchase"] > 0]["days_to_next_purchase"].clip(upper=180).hist(
    bins=60, figsize=(12,4)
)
plt.title("Distribution sans les 0 (cap 180j)")
plt.xlabel("Jours")
plt.show()

In [ ]:
# Agréger par ticket d'abord
ticket_df = df.groupby(
    ["anonymized_card_code", "anonymized_Ticket_ID", "transactionDate"]
).agg(
    total_spent    = ("salesVatEUR", "sum"),
    total_discount = ("discountEUR", "sum"),
    total_qty      = ("quantity", "sum"),
    nb_items       = ("materialCode", "nunique"),
    has_return     = ("is_return", "max"),
    channel        = ("channel", "first"),
    store_city     = ("store_city", "first"),
    Axe_Desc       = ("Axe_Desc", lambda x: x.mode()[0]),
    Market_Desc    = ("Market_Desc", lambda x: x.mode()[0]),
    brand          = ("brand", lambda x: x.mode()[0]),
    gender         = ("gender", "first"),
    age            = ("age", "first"),
    age_category   = ("age_category", "first"),
    RFM_Segment_ID = ("RFM_Segment_ID", "first"),
    status         = ("status", "first"),
    days_since_sub = ("days_since_sub", "first"),
    has_first_purchase_data = ("has_first_purchase_data", "first"),
    days_to_next_purchase   = ("days_to_next_purchase", "min"),
).reset_index()

# Variable cible en 3 classes
def label_next(d):
    if pd.isna(d):    return "no_next"      # dernier achat connu
    elif d <= 30:     return "court"        # rachète dans le mois
    elif d <= 90:     return "moyen"        # rachète dans le trimestre
    else:             return "long"         # rachète après 90j ou peu actif

ticket_df["target"] = ticket_df["days_to_next_purchase"].apply(label_next)
print(ticket_df["target"].value_counts())

In [ ]:
# Variable cible en 4 classes (on garde no_next)
def label_next(d):
    if pd.isna(d):  return "no_next"
    elif d <= 30:   return "court"
    elif d <= 90:   return "moyen"
    else:           return "long"

ticket_df["target"] = ticket_df["days_to_next_purchase"].apply(label_next)
print(ticket_df["target"].value_counts())

# Encodage des colonnes catégorielles
from sklearn.preprocessing import LabelEncoder

cat_cols = ["channel", "store_city", "Axe_Desc", "Market_Desc", "brand", "age_category"]
le = {}
for col in cat_cols:
    le[col] = LabelEncoder()
    ticket_df[col] = le[col].fit_transform(ticket_df[col].astype(str))

# Imputer RFM_Segment_ID (97 NaN restants)
ticket_df["RFM_Segment_ID"] = ticket_df["RFM_Segment_ID"].fillna(
    ticket_df["RFM_Segment_ID"].median()
)

# Définition des features et de la cible
features = [
    "total_spent", "total_discount", "total_qty", "nb_items", "has_return",
    "gender", "age", "RFM_Segment_ID", "status", "days_since_sub",
    "has_first_purchase_data",
    "channel", "store_city", "Axe_Desc", "Market_Desc", "brand",
    "age_category"
]

X = ticket_df[features]
y = ticket_df["target"]

# Split train / test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entraînement Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Évaluation
print(classification_report(y_test, rf.predict(X_test)))

In [ ]:
# Historique par client sur l'ensemble du df
client_history = df.groupby("anonymized_card_code").agg(
    nb_tickets_total    = ("anonymized_Ticket_ID", "nunique"),
    avg_basket          = ("salesVatEUR", "sum"),
    avg_interachat      = ("transactionDate", lambda x: x.sort_values().diff().dt.days.mean()),
    return_rate         = ("is_return", "mean"),
    nb_categories       = ("Axe_Desc", "nunique"),
    nb_brands           = ("brand", "nunique"),
).reset_index()

# Fusionner sur ticket_df
ticket_df = ticket_df.merge(client_history, on="anonymized_card_code", how="left")

features = features + [
    "nb_tickets_total", "avg_basket", "avg_interachat",
    "return_rate", "nb_categories", "nb_brands"
]

In [ ]:
# Option 1 — 3 classes : court / rare / no_next
def label_next(d):
    if pd.isna(d):  return "no_next"
    elif d <= 30:   return "court"
    else:           return "rare"

ticket_df["target"] = ticket_df["days_to_next_purchase"].apply(label_next)
print(ticket_df["target"].value_counts())

# Ré-encodage (repart de zéro sur ticket_df propre)
cat_cols = ["channel", "store_city", "Axe_Desc", "Market_Desc", "brand", "age_category"]
le = {}
for col in cat_cols:
    le[col] = LabelEncoder()
    ticket_df[col] = le[col].fit_transform(ticket_df[col].astype(str))

ticket_df["RFM_Segment_ID"] = ticket_df["RFM_Segment_ID"].fillna(
    ticket_df["RFM_Segment_ID"].median()
)

features = [
    "total_spent", "total_discount", "total_qty", "nb_items", "has_return",
    "gender", "age", "RFM_Segment_ID", "status", "days_since_sub",
    "has_first_purchase_data",
    "channel", "store_city", "Axe_Desc", "Market_Desc", "brand",
    "age_category"
]

X = ticket_df[features]
y = ticket_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test)))

In [ ]:
# Historique client
client_history = df.groupby("anonymized_card_code").agg(
    nb_tickets_total = ("anonymized_Ticket_ID", "nunique"),
    avg_basket       = ("salesVatEUR", "sum"),
    avg_interachat   = ("transactionDate", lambda x: x.sort_values().diff().dt.days.mean()),
    return_rate      = ("is_return", "mean"),
    nb_categories    = ("Axe_Desc", "nunique"),
    nb_brands        = ("brand", "nunique"),
).reset_index()

ticket_df = ticket_df.merge(client_history, on="anonymized_card_code", how="left")

# Vérification — doit afficher les nouvelles colonnes
print(ticket_df[["nb_tickets_total","avg_basket","avg_interachat","return_rate"]].describe())

# Features redéfinies proprement
features = [
    "total_spent", "total_discount", "total_qty", "nb_items", "has_return",
    "gender", "age", "RFM_Segment_ID", "status", "days_since_sub",
    "has_first_purchase_data",
    "channel", "store_city", "Axe_Desc", "Market_Desc", "brand", "age_category",
    "nb_tickets_total", "avg_basket", "avg_interachat", "return_rate",
    "nb_categories", "nb_brands"
]
print("Nb features :", len(features))  # doit afficher 23

X = ticket_df[features]
y = ticket_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test)))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importances = pd.Series(rf.feature_importances_, index=features)
importances.sort_values().tail(15).plot(kind="barh", figsize=(10, 6))
plt.title("Importance des features — Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# ── ÉTAPE 1 : Recréer ticket_df proprement avec toutes les colonnes nécessaires ──
ticket_df = df.groupby(
    ["anonymized_card_code", "anonymized_Ticket_ID", "transactionDate"]
).agg(
    total_spent       = ("salesVatEUR",    "sum"),
    total_discount    = ("discountEUR",    "sum"),
    total_qty         = ("quantity",       "sum"),
    nb_items          = ("materialCode",   "nunique"),
    has_return        = ("is_return",      "max"),
    channel           = ("channel",        "first"),
    store_type_app    = ("store_type_app", "first"),   # ajouté
    store_city        = ("store_city",     "first"),
    Axe_Desc          = ("Axe_Desc",       lambda x: x.mode()[0]),
    Market_Desc       = ("Market_Desc",    lambda x: x.mode()[0]),
    brand             = ("brand",          lambda x: x.mode()[0]),
    gender            = ("gender",         "first"),
    age               = ("age",            "first"),
    age_category      = ("age_category",   "first"),
    RFM_Segment_ID    = ("RFM_Segment_ID", "first"),
    status            = ("status",         "first"),
    days_since_sub    = ("days_since_sub", "first"),
    has_first_purchase_data = ("has_first_purchase_data", "first"),
    days_to_next_purchase   = ("days_to_next_purchase",   "min"),
).reset_index()

print("ticket_df shape :", ticket_df.shape)

# ── ÉTAPE 2 : Variable cible ──
def label_next(d):
    if pd.isna(d):  return "no_next"
    elif d <= 30:   return "court"
    else:           return "rare"

ticket_df["target"] = ticket_df["days_to_next_purchase"].apply(label_next)
print(ticket_df["target"].value_counts())

# ── ÉTAPE 3 : Encodage ──
from sklearn.preprocessing import LabelEncoder

cat_cols = ["channel", "store_type_app", "store_city",
            "Axe_Desc", "Market_Desc", "brand", "age_category"]
le = {}
for col in cat_cols:
    le[col] = LabelEncoder()
    ticket_df[col] = le[col].fit_transform(ticket_df[col].astype(str))

ticket_df["RFM_Segment_ID"] = ticket_df["RFM_Segment_ID"].fillna(
    ticket_df["RFM_Segment_ID"].median()
)

# ── ÉTAPE 4 : Features historiques par client ──
client_history = df.groupby("anonymized_card_code").agg(
    nb_tickets_total  = ("anonymized_Ticket_ID", "nunique"),
    avg_basket        = ("salesVatEUR",           "sum"),
    avg_interachat    = ("transactionDate",        lambda x: x.sort_values().diff().dt.days.mean()),
    return_rate       = ("is_return",              "mean"),
    nb_categories     = ("Axe_Desc",              "nunique"),
    nb_brands         = ("brand",                 "nunique"),
).reset_index()

ticket_df = ticket_df.merge(client_history, on="anonymized_card_code", how="left")

# ── ÉTAPE 5 : Profil client complet (1 ligne = 1 client) ──
client_profile = ticket_df.sort_values("transactionDate").groupby(
    "anonymized_card_code"
).last().reset_index()[[
    "anonymized_card_code", "transactionDate",
    "gender", "age", "age_category", "status", "RFM_Segment_ID",
    "days_since_sub", "has_first_purchase_data",
    "nb_tickets_total", "avg_basket", "avg_interachat",
    "return_rate", "nb_categories", "nb_brands", "target"
]]

client_habits = ticket_df.groupby("anonymized_card_code").agg(
    axe_favori        = ("Axe_Desc",        lambda x: x.mode()[0]),
    market_favori     = ("Market_Desc",     lambda x: x.mode()[0]),
    brand_favorite    = ("brand",           lambda x: x.mode()[0]),
    canal_favori      = ("channel",         lambda x: x.mode()[0]),
    store_type_favori = ("store_type_app",  lambda x: x.mode()[0]),
    avg_salesVat      = ("total_spent",     "mean"),
    max_salesVat      = ("total_spent",     "max"),
    total_revenue     = ("total_spent",     "sum"),
).reset_index()

client_full = client_profile.merge(client_habits, on="anonymized_card_code", how="left")

print("client_full shape :", client_full.shape)
print(client_full.head(3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error

features_client = [
    "gender", "age", "status", "RFM_Segment_ID", "days_since_sub",
    "has_first_purchase_data", "nb_tickets_total", "avg_basket",
    "avg_interachat", "return_rate", "nb_categories", "nb_brands",
    "axe_favori", "market_favori", "brand_favorite",
    "canal_favori", "store_type_favori"
]

# Imputer les NaN restants
client_full[features_client] = client_full[features_client].fillna(
    client_full[features_client].median(numeric_only=True)
)

X = client_full[features_client]

# ── Modèle 2 : prochain axe produit ──
rf_axe = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 random_state=42, n_jobs=-1)
X_tr, X_te, y_tr, y_te = train_test_split(X, client_full["axe_favori"],
                                           test_size=0.2, random_state=42)
rf_axe.fit(X_tr, y_tr)
print("── Modèle 2 : axe produit ──")
print(classification_report(y_te, rf_axe.predict(X_te)))

# ── Modèle 3 : canal de conversion ──
rf_canal = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                   random_state=42, n_jobs=-1)
X_tr, X_te, y_tr, y_te = train_test_split(X, client_full["canal_favori"],
                                           test_size=0.2, random_state=42)
rf_canal.fit(X_tr, y_tr)
print("── Modèle 3 : canal ──")
print(classification_report(y_te, rf_canal.predict(X_te)))

# ── Modèle 4 : valeur attendue (régression) ──
rf_val = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
X_tr, X_te, y_tr, y_te = train_test_split(X, client_full["avg_salesVat"],
                                           test_size=0.2, random_state=42)
rf_val.fit(X_tr, y_tr)
preds = rf_val.predict(X_te)
print("── Modèle 4 : valeur attendue ──")
print(f"MAE : {mean_absolute_error(y_te, preds):.2f}€")
print(f"Valeur moyenne réelle   : {y_te.mean():.2f}€")
print(f"Valeur moyenne prédite  : {preds.mean():.2f}€")

In [ ]:
# ── Scoring client — 4 classes de valeur ──
client_score = ticket_df.groupby("anonymized_card_code").agg(
    frequence     = ("anonymized_Ticket_ID", "nunique"),
    valeur_totale = ("total_spent", "sum"),
    panier_moyen  = ("total_spent", "mean"),
).reset_index()

# Rank percentile sur chaque dimension (évite le problème de doublons)
for col in ["frequence", "valeur_totale", "panier_moyen"]:
    client_score[f"{col}_q"] = pd.cut(
        client_score[col].rank(pct=True),
        bins=[0, 0.25, 0.50, 0.75, 1.0],
        labels=[1, 2, 3, 4],
        include_lowest=True
    ).astype(int)

client_score["score_global"] = (
    client_score["frequence_q"] +
    client_score["valeur_totale_q"] +
    client_score["panier_moyen_q"]
)

# 4 segments — score va de 3 à 12
client_score["segment_valeur"] = pd.cut(
    client_score["score_global"],
    bins=[2, 5, 7, 9, 12],
    labels=["bronze", "silver", "gold", "platinum"],
    include_lowest=True
)

print(client_score["segment_valeur"].value_counts())
print("\n")
print(client_score.groupby("segment_valeur")[
    ["frequence", "valeur_totale", "panier_moyen"]
].mean().round(2))

In [ ]:
# ════════════════════════════════════════════════════════
# BLOC COMPLET — Scoring client + Analyse par segment
# ════════════════════════════════════════════════════════

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# ── 1. Recréer ticket_df proprement ──
ticket_df = df.groupby(
    ["anonymized_card_code", "anonymized_Ticket_ID", "transactionDate"]
).agg(
    total_spent       = ("salesVatEUR",    "sum"),
    total_discount    = ("discountEUR",    "sum"),
    total_qty         = ("quantity",       "sum"),
    nb_items          = ("materialCode",   "nunique"),
    has_return        = ("is_return",      "max"),
    channel           = ("channel",        "first"),
    store_type_app    = ("store_type_app", "first"),
    store_city        = ("store_city",     "first"),
    Axe_Desc          = ("Axe_Desc",       lambda x: x.mode()[0]),
    Market_Desc       = ("Market_Desc",    lambda x: x.mode()[0]),
    brand             = ("brand",          lambda x: x.mode()[0]),
    gender            = ("gender",         "first"),
    age               = ("age",            "first"),
    age_category      = ("age_category",   "first"),
    RFM_Segment_ID    = ("RFM_Segment_ID", "first"),
    status            = ("status",         "first"),
    days_since_sub    = ("days_since_sub", "first"),
    has_first_purchase_data = ("has_first_purchase_data", "first"),
    days_to_next_purchase   = ("days_to_next_purchase",   "min"),
).reset_index()

print("ticket_df shape :", ticket_df.shape)

# ── 2. Variable cible ──
def label_next(d):
    if pd.isna(d):  return "no_next"
    elif d <= 30:   return "court"
    else:           return "rare"

ticket_df["target"] = ticket_df["days_to_next_purchase"].apply(label_next)

# ── 3. Encodage ──
cat_cols = ["channel", "store_type_app", "store_city",
            "Axe_Desc", "Market_Desc", "brand", "age_category"]
le = {}
for col in cat_cols:
    le[col] = LabelEncoder()
    ticket_df[col] = le[col].fit_transform(ticket_df[col].astype(str))

ticket_df["RFM_Segment_ID"] = ticket_df["RFM_Segment_ID"].fillna(
    ticket_df["RFM_Segment_ID"].median()
)

# ── 4. Features historiques par client ──
client_history = df.groupby("anonymized_card_code").agg(
    nb_tickets_total = ("anonymized_Ticket_ID", "nunique"),
    avg_basket       = ("salesVatEUR",           "sum"),
    avg_interachat   = ("transactionDate",        lambda x: x.sort_values().diff().dt.days.mean()),
    return_rate      = ("is_return",              "mean"),
    nb_categories    = ("Axe_Desc",              "nunique"),
    nb_brands        = ("brand",                 "nunique"),
).reset_index()

ticket_df = ticket_df.merge(client_history, on="anonymized_card_code", how="left")

# ── 5. Scoring client RFM maison ──
client_score = ticket_df.groupby("anonymized_card_code").agg(
    frequence     = ("anonymized_Ticket_ID", "nunique"),
    valeur_totale = ("total_spent",          "sum"),
    panier_moyen  = ("total_spent",          "mean"),
).reset_index()

for col in ["frequence", "valeur_totale", "panier_moyen"]:
    client_score[f"{col}_q"] = pd.cut(
        client_score[col].rank(pct=True),
        bins=[0, 0.25, 0.50, 0.75, 1.0],
        labels=[1, 2, 3, 4],
        include_lowest=True
    ).astype(int)

client_score["score_global"] = (
    client_score["frequence_q"] +
    client_score["valeur_totale_q"] +
    client_score["panier_moyen_q"]
)

client_score["segment_valeur"] = pd.cut(
    client_score["score_global"],
    bins=[2, 5, 7, 9, 12],
    labels=["bronze", "silver", "gold", "platinum"],
    include_lowest=True
)

print("Segments :", client_score["segment_valeur"].value_counts().to_dict())

# ── 6. Fusion segment + décodage labels ──
if "segment_valeur" in ticket_df.columns:
    ticket_df = ticket_df.drop(columns=["segment_valeur"])

ticket_df = ticket_df.merge(
    client_score[["anonymized_card_code", "segment_valeur"]],
    on="anonymized_card_code", how="left"
)

for col in ["Axe_Desc", "Market_Desc", "brand", "channel", "store_type_app"]:
    ticket_df[f"{col}_label"] = le[col].inverse_transform(
        ticket_df[col].fillna(0).astype(int)
    )

print("ticket_df final shape :", ticket_df.shape)

# ── 7. Analyse par segment ──
print("\n=== PROFIL DÉMOGRAPHIQUE ===")
print(ticket_df.groupby("segment_valeur").agg(
    age_moyen      = ("age",            "mean"),
    statut_moyen   = ("status",         "mean"),
    rfm_moyen      = ("RFM_Segment_ID", "mean"),
    anciennete_moy = ("days_since_sub", "mean"),
    panier_moyen   = ("total_spent",    "mean"),
    taux_retour    = ("has_return",     "mean"),
).round(2))

print("\n=== TOP 3 CATÉGORIES PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "Axe_Desc_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(3).to_string())

print("\n=== TOP 3 MARCHÉS PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "Market_Desc_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(3).to_string())

print("\n=== CANAL PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "channel_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(2).to_string())

print("\n=== TOP 3 MARQUES PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "brand_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(3).to_string())

In [ ]:
print("\n=== TOP 3 MARCHÉS PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "Market_Desc_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(3).to_string())

print("\n=== CANAL PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "channel_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(2).to_string())

print("\n=== TOP 3 MARQUES PAR SEGMENT ===")
print(ticket_df.groupby(["segment_valeur", "brand_label"])
    .size().reset_index(name="nb")
    .sort_values(["segment_valeur", "nb"], ascending=[True, False])
    .groupby("segment_valeur").head(3).to_string())

In [ ]:
# Proportion de clients par segment qui achètent chaque marché
# (pas le volume, mais le % de clients uniques)
print(
    ticket_df.groupby("segment_valeur").apply(
        lambda x: x.groupby("Market_Desc_label")["anonymized_card_code"]
        .nunique() / x["anonymized_card_code"].nunique() * 100
    ).round(1)
)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd

# ── Préparer les séquences par client et segment ──
# On travaille sur ticket_df trié par date
ticket_df_sorted = ticket_df.sort_values(
    ["anonymized_card_code", "transactionDate"]
)

# Créer une colonne combinée "profil_achat" = Axe + Market + Brand
ticket_df_sorted["profil_achat"] = (
    ticket_df_sorted["Axe_Desc_label"] + "_" +
    ticket_df_sorted["Market_Desc_label"] + "_" +
    ticket_df_sorted["brand_label"]
)

# ── Construire la matrice panier par client (one-hot) ──
# On travaille segment par segment pour des règles ciblées
resultats = {}

for segment in ["bronze", "silver", "gold", "platinum"]:
    print(f"\nTraitement segment : {segment}")
    
    clients_seg = ticket_df_sorted[
        ticket_df_sorted["segment_valeur"] == segment
    ]["anonymized_card_code"].unique()
    
    # Tickets par client — liste des profils achetés
    basket = ticket_df_sorted[
        ticket_df_sorted["anonymized_card_code"].isin(clients_seg)
    ].groupby(["anonymized_card_code", "anonymized_Ticket_ID"])["profil_achat"] \
     .apply(lambda x: list(x.unique())) \
     .reset_index()
    
    # One-hot encoding par client (pas par ticket — on veut les co-occurrences)
    basket_client = ticket_df_sorted[
        ticket_df_sorted["anonymized_card_code"].isin(clients_seg)
    ].groupby(["anonymized_card_code", "profil_achat"]).size() \
     .unstack(fill_value=0).clip(upper=1)
    
    # Apriori — support min 2% pour avoir des règles significatives
    frequent = apriori(
        basket_client.astype(bool),
        min_support=0.02,
        use_colnames=True,
        max_len=2
    )
    
    if len(frequent) == 0:
        print(f"  Pas de règles pour {segment}")
        continue
    
    # Règles d'association
    rules = association_rules(
        frequent,
        metric="confidence",
        min_threshold=0.3
    )
    
    rules["segment"] = segment
    resultats[segment] = rules
    print(f"  {len(rules)} règles trouvées")

# ── Afficher les top 5 règles par segment ──
for segment, rules in resultats.items():
    print(f"\n{'='*50}")
    print(f"TOP 5 RÈGLES — {segment.upper()}")
    print(f"{'='*50}")
    top = rules.sort_values("confidence", ascending=False).head(5)
    for _, row in top.iterrows():
        ant = list(row["antecedents"])[0]
        con = list(row["consequents"])[0]
        conf = row["confidence"] * 100
        supp = row["support"] * 100
        lift = row["lift"]
        print(f"  {ant}")
        print(f"  → {con}")
        print(f"  Confiance: {conf:.1f}% | Support: {supp:.1f}% | Lift: {lift:.2f}")
        print()

In [ ]:
# Corriger la faute de frappe MAEK UP → MAKE UP dans df original
df["Axe_Desc"] = df["Axe_Desc"].str.strip().str.upper()
df["Axe_Desc"] = df["Axe_Desc"].replace("MAEK UP", "MAKE UP")

# Vérification
print(df["Axe_Desc"].value_counts())

In [ ]:
# ── Matrice de transition : achat N → achat N+1 ──
ticket_df_sorted = ticket_df.sort_values(["anonymized_card_code", "transactionDate"])

# Achat suivant par client
ticket_df_sorted["next_axe_label"] = ticket_df_sorted.groupby(
    "anonymized_card_code"
)["Axe_Desc_label"].shift(-1)

# Filtrer les catégories parasites et les derniers achats (pas de next)
mask = (
    ~ticket_df_sorted["Axe_Desc_label"].isin(["NOT SPECIFIED", "OTHERS", "inconnu"]) &
    ~ticket_df_sorted["next_axe_label"].isin(["NOT SPECIFIED", "OTHERS", "inconnu"]) &
    ticket_df_sorted["next_axe_label"].notna()
)
transitions = ticket_df_sorted[mask]

# Matrice par segment
for segment in ["bronze", "silver", "gold", "platinum"]:
    seg_data = transitions[transitions["segment_valeur"] == segment]
    
    matrix = pd.crosstab(
        seg_data["Axe_Desc_label"],
        seg_data["next_axe_label"],
        normalize="index"  # % par ligne = "après X, combien achètent Y"
    ).round(3) * 100
    
    print(f"\n{'='*50}")
    print(f"MATRICE DE TRANSITION — {segment.upper()} (en %)")
    print(f"{'='*50}")
    print(matrix.to_string())

In [ ]:
# Valeur moyenne du ticket N+1 selon la transition
ticket_df_sorted["next_montant"] = ticket_df_sorted.groupby(
    "anonymized_card_code"
)["total_spent"].shift(-1)

transition_value = ticket_df_sorted.dropna(
    subset=["next_axe_label", "next_montant"]
).groupby(["segment_valeur", "Axe_Desc_label", "next_axe_label"]).agg(
    nb_transitions = ("anonymized_card_code", "count"),
    valeur_moyenne = ("next_montant", "mean"),
    valeur_totale  = ("next_montant", "sum"),
).reset_index().round(2)

# Top 5 transitions les plus rentables par segment
print(transition_value[
    ~transition_value["Axe_Desc_label"].isin(["NOT SPECIFIED", "OTHERS", "inconnu"]) &
    ~transition_value["next_axe_label"].isin(["NOT SPECIFIED", "OTHERS", "inconnu"])
].sort_values(["segment_valeur", "valeur_moyenne"], ascending=[True, False])
 .groupby("segment_valeur").head(5)
 .to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Données des matrices
categories = ['FRAGRANCE', 'HAIRCARE', 'MAKE UP', 'SKINCARE']

matrices = {
    'Bronze': np.array([
        [60.8,  3.1,  7.6, 20.8],
        [ 1.4, 27.0, 40.5, 31.1],
        [ 1.3,  2.8, 75.9, 20.0],
        [ 1.6,  3.6, 35.2, 59.6],
    ]),
    'Silver': np.array([
        [29.8,  2.7, 36.7, 30.7],
        [ 6.2, 24.9, 41.1, 27.7],
        [ 8.3,  2.8, 69.2, 19.6],
        [ 9.6,  4.0, 40.7, 45.7],
    ]),
    'Gold': np.array([
        [33.6,  3.7, 40.7, 22.1],
        [12.6, 18.7, 45.8, 22.9],
        [13.4,  3.7, 64.5, 18.4],
        [15.1,  4.2, 40.8, 39.9],
    ]),
    'Platinum': np.array([
        [45.1,  4.0, 34.6, 16.2],
        [16.8, 17.7, 42.5, 23.0],
        [17.7,  5.2, 58.8, 18.3],
        [19.3,  6.2, 40.4, 34.0],
    ]),
}

colors = {
    'Bronze':   '#8B7355',
    'Silver':   '#185FA5',
    'Gold':     '#BA7517',
    'Platinum': '#534AB7',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for idx, (segment, matrix) in enumerate(matrices.items()):
    ax = axes[idx]

    im = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=70, aspect='auto')

    # Annotations
    for i in range(4):
        for j in range(4):
            val = matrix[i, j]
            color = 'white' if val >= 40 else '#1a1a2e'
            weight = 'bold' if i == j else 'normal'
            # Encadrer la diagonale
            if i == j:
                ax.add_patch(plt.Rectangle(
                    (j - 0.5, i - 0.5), 1, 1,
                    fill=False, edgecolor=colors[segment],
                    linewidth=2.5
                ))
            ax.text(j, i, f'{val:.1f}%',
                    ha='center', va='center',
                    fontsize=11, color=color, fontweight=weight)

    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_yticklabels(categories, fontsize=10)
    ax.set_xlabel('Prochain achat (N+1)', fontsize=10, labelpad=8)
    ax.set_ylabel('Achat actuel (N)', fontsize=10, labelpad=8)
    ax.set_title(segment, fontsize=14, fontweight='bold',
                 color=colors[segment], pad=12)

    plt.colorbar(im, ax=ax, shrink=0.8, label='%')

fig.suptitle(
    'Matrices de transition — Probabilité du prochain achat par segment',
    fontsize=15, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig('matrices_transition.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print("Fichier sauvegardé : matrices_transition.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Données ──────────────────────────────────────────
segments  = ['Bronze', 'Silver', 'Gold', 'Platinum']
colors    = ['#8B7355', '#185FA5', '#BA7517', '#534AB7']
alphas    = [0.75, 0.80, 0.85, 1.0]

transitions = {
    'Bronze': [
        ('SKINCARE → FRAGRANCE',  16,    87.20,   1395),
        ('MAKE UP → FRAGRANCE',   22,    23.85,    525),
        ('FRAGRANCE → HAIRCARE',   4,    22.92,     92),
        ('HAIRCARE → FRAGRANCE',   1,    25.20,     25),
        ('FRAGRANCE → MAKE UP',   15,    28.00,    420),
    ],
    'Silver': [
        ('SKINCARE → FRAGRANCE', 272,    50.51,  13738),
        ('MAKE UP → FRAGRANCE',  167,    56.75,   9477),
        ('MAEK UP → FRAGRANCE',  259,    56.19,  14552),
        ('HAIRCARE → FRAGRANCE',  25,    39.88,    997),
        ('MAKE UP → HAIRCARE',    57,    38.04,   2168),
    ],
    'Gold': [
        ('SKINCARE → FRAGRANCE',  978,    77.36,  75663),
        ('MAKE UP → FRAGRANCE',   636,    82.75,  52631),
        ('FRAGRANCE → FRAGRANCE',1303,    59.81,  77937),
        ('HAIRCARE → FRAGRANCE',  137,    79.35,  10871),
        ('MAEK UP → FRAGRANCE',  1204,    81.53,  98163),
    ],
    'Platinum': [
        ('FRAGRANCE → FRAGRANCE',8590,   117.65, 1010601),
        ('SKINCARE → FRAGRANCE', 3459,   116.24,  402072),
        ('MAKE UP → FRAGRANCE',  2257,   119.11,  268827),
        ('MAEK UP → FRAGRANCE',  4870,   119.07,  579856),
        ('HAIRCARE → FRAGRANCE',  813,   112.78,   91686),
    ],
}

# ── Figure 3 graphiques ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Rentabilité des transitions par segment',
             fontsize=16, fontweight='bold', y=1.02)

# ── Graphique 1 : Panier moyen par transition ────────
ax1 = axes[0]
x = np.arange(5)
width = 0.2

for i, (seg, col, alp) in enumerate(zip(segments, colors, alphas)):
    vals = [t[2] for t in transitions[seg]]
    bars = ax1.bar(x + i * width, vals, width, label=seg,
                   color=col, alpha=alp, edgecolor='white', linewidth=0.5)

ax1.set_title('Panier moyen par transition (€)', fontsize=12, fontweight='bold')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels([
    'SKINCARE\n→ FRAGRANCE', 'MAKE UP\n→ FRAGRANCE',
    'FRAGRANCE\n→ FRAGRANCE', 'HAIRCARE\n→ FRAGRANCE',
    'MAKE UP\n→ HAIRCARE'
], fontsize=8)
ax1.set_ylabel('Panier moyen (€)')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 140)

# ── Graphique 2 : Bubble chart valeur totale × panier ─
ax2 = axes[1]
for seg, col in zip(segments, colors):
    for label, nb, panier, total in transitions[seg]:
        size = np.sqrt(total) * 0.8
        ax2.scatter(panier, nb, s=size, color=col,
                    alpha=0.65, edgecolors=col, linewidth=1)

# Légende segments
handles = [mpatches.Patch(color=c, label=s)
           for s, c in zip(segments, colors)]
ax2.legend(handles=handles, fontsize=9)
ax2.set_xlabel('Panier moyen (€)')
ax2.set_ylabel('Nombre de transitions')
ax2.set_title('Volume × Valeur\n(taille bulle = valeur totale)',
             fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

# ── Graphique 3 : Valeur totale top transitions ───────
ax3 = axes[2]
top_trans = [
    ('Plat. FRAGRANCE→FRAGRANCE', 1010601, '#534AB7'),
    ('Plat. MAEK UP→FRAGRANCE',   579856,  '#534AB7'),
    ('Plat. SKINCARE→FRAGRANCE',  402072,  '#534AB7'),
    ('Plat. MAKE UP→FRAGRANCE',   268827,  '#534AB7'),
    ('Gold FRAGRANCE→FRAGRANCE',   98163,  '#BA7517'),
    ('Gold SKINCARE→FRAGRANCE',    77937,  '#BA7517'),
    ('Gold MAEK UP→FRAGRANCE',     75663,  '#BA7517'),
    ('Silver SKINCARE→FRAGRANCE',  14552,  '#185FA5'),
    ('Silver MAKE UP→FRAGRANCE',   13738,  '#185FA5'),
    ('Bronze SKINCARE→FRAGRANCE',   1395,  '#8B7355'),
]

labels_t = [t[0] for t in top_trans]
values_t = [t[1] for t in top_trans]
colors_t = [t[2] for t in top_trans]

bars = ax3.barh(labels_t[::-1], values_t[::-1],
                color=colors_t[::-1], edgecolor='white',
                linewidth=0.5, alpha=0.85)

for bar, val in zip(bars, values_t[::-1]):
    ax3.text(bar.get_width() + 5000, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}€', va='center', fontsize=8)

ax3.set_title('Top 10 transitions — valeur totale générée',
             fontsize=12, fontweight='bold')
ax3.set_xlabel('Valeur totale (€)')
ax3.grid(axis='x', alpha=0.3)
ax3.set_xlim(0, 1200000)

plt.tight_layout()
plt.savefig('transitions_rentabilite.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print("Fichier sauvegardé : transitions_rentabilite.png")

In [ ]:
segments = ['Bronze', 'Silver', 'Gold', 'Platinum']
colors   = ['#8B7355', '#185FA5', '#BA7517', '#534AB7']

transitions_labels = [
    'SKINCARE\n→FRAGRANCE',
    'MAKE UP\n→FRAGRANCE',
    'FRAGRANCE\n→FRAGRANCE',
    'HAIRCARE\n→FRAGRANCE',
    'MAKE UP\n→HAIRCARE'
]

# Panier moyen par transition et segment
paniers = {
    'Bronze':   [87.20, 23.85, 22.92, 25.20, 28.00],
    'Silver':   [50.51, 56.75, 56.19, 39.88, 38.04],
    'Gold':     [77.36, 82.75, 59.81, 79.35, 81.53],
    'Platinum': [116.24,119.11,117.65,112.78,119.07],
}

# Nombre de transitions par type et segment
volumes = {
    'Bronze':   [16,   22,   4,    1,   15],
    'Silver':   [272,  167,  259,  25,  57],
    'Gold':     [978,  636,  1303, 137, 1204],
    'Platinum': [3459, 2257, 8590, 813, 4870],
}

# Valeur totale par transition et segment
valeurs = {
    'Bronze':   [1395,   525,    92,    25,   420],
    'Silver':   [13738,  9477,  14552,  997,  2168],
    'Gold':     [75663,  52631, 77937, 10871, 98163],
    'Platinum': [402072, 268827,1010601,91686,579856],
}

x = np.arange(len(transitions_labels))

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Évolution des transitions par segment',
             fontsize=16, fontweight='bold', y=1.02)

# ── Graphique 1 : Courbes panier moyen ──────────────
ax1 = axes[0]
for seg, col in zip(segments, colors):
    vals = paniers[seg]
    ax1.plot(x, vals, marker='o', color=col, linewidth=2.5,
             markersize=7, label=seg)
    # Valeur sur chaque point
    for i, v in enumerate(vals):
        ax1.annotate(f'{v:.0f}€', (x[i], v),
                     textcoords='offset points',
                     xytext=(0, 10), ha='center',
                     fontsize=8, color=col)

# Courbe moyenne globale
moy_globale = [np.mean([paniers[s][i] for s in segments])
               for i in range(len(transitions_labels))]
ax1.plot(x, moy_globale, marker='D', color='black',
         linewidth=2, linestyle='--', markersize=6,
         label='Moyenne globale', alpha=0.7)
for i, v in enumerate(moy_globale):
    ax1.annotate(f'{v:.0f}€', (x[i], v),
                 textcoords='offset points',
                 xytext=(0, -16), ha='center',
                 fontsize=8, color='black', alpha=0.7)

ax1.set_title('Panier moyen par transition (€)',
              fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(transitions_labels, fontsize=9)
ax1.set_ylabel('Panier moyen (€)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 150)

# ── Graphique 2 : Courbes volume de transitions ─────
ax2 = axes[1]
for seg, col in zip(segments, colors):
    vals = volumes[seg]
    ax2.plot(x, vals, marker='o', color=col, linewidth=2.5,
             markersize=7, label=seg)
    for i, v in enumerate(vals):
        ax2.annotate(f'{v:,}', (x[i], v),
                     textcoords='offset points',
                     xytext=(0, 10), ha='center',
                     fontsize=8, color=col)

# Courbe moyenne globale
moy_vol = [np.mean([volumes[s][i] for s in segments])
           for i in range(len(transitions_labels))]
ax2.plot(x, moy_vol, marker='D', color='black',
         linewidth=2, linestyle='--', markersize=6,
         label='Moyenne globale', alpha=0.7)
for i, v in enumerate(moy_vol):
    ax2.annotate(f'{v:.0f}', (x[i], v),
                 textcoords='offset points',
                 xytext=(0, -16), ha='center',
                 fontsize=8, color='black', alpha=0.7)

ax2.set_title('Volume de transitions par type',
              fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(transitions_labels, fontsize=9)
ax2.set_ylabel('Nombre de transitions')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# ── Graphique 3 : Courbes valeur totale ─────────────
ax3 = axes[2]
for seg, col in zip(segments, colors):
    vals = valeurs[seg]
    ax3.plot(x, vals, marker='o', color=col, linewidth=2.5,
             markersize=7, label=seg)
    for i, v in enumerate(vals):
        label = f'{v/1000:.0f}k€' if v >= 1000 else f'{v}€'
        ax3.annotate(label, (x[i], v),
                     textcoords='offset points',
                     xytext=(0, 10), ha='center',
                     fontsize=8, color=col)

# Courbe moyenne globale
moy_val = [np.mean([valeurs[s][i] for s in segments])
           for i in range(len(transitions_labels))]
ax3.plot(x, moy_val, marker='D', color='black',
         linewidth=2, linestyle='--', markersize=6,
         label='Moyenne globale', alpha=0.7)
for i, v in enumerate(moy_val):
    ax3.annotate(f'{v/1000:.0f}k€', (x[i], v),
                 textcoords='offset points',
                 xytext=(0, -16), ha='center',
                 fontsize=8, color='black', alpha=0.7)

ax3.set_title('Valeur totale générée par transition (€)',
              fontsize=12, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(transitions_labels, fontsize=9)
ax3.set_ylabel('Valeur totale (€)')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('transitions_courbes.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()
print("Fichier sauvegardé : transitions_courbes.png")

In [ ]:
# ── Vérification de l'état du df ──
print("Shape df :", df.shape)
print("\nValeurs uniques age_generation :")
print(df["age_generation"].value_counts())
print("\nValeurs uniques brand (top 15) :")
print(df["brand"].value_counts().head(15))
print("\nValeurs uniques Axe_Desc :")
print(df["Axe_Desc"].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Filtrer les valeurs parasites ──
df_clean = df[
    ~df["Axe_Desc"].isin(["NOT SPECIFIED", "OTHERS"]) &
    (df["age_generation"] != "inconnu") &
    (df["salesVatEUR"] > 0)  # exclure retours pour cette analyse
].copy()

print("Lignes après filtre :", df_clean.shape[0])
print("Générations :", df_clean["age_generation"].unique())

# ── Ordre des générations ──
gen_order = ["genz", "geny", "gena", "babyboomers"]
gen_labels = {"genz": "Gen Z\n(~1997-2012)", "geny": "Gen Y\n(~1981-1996)",
              "gena": "Gen A / Alpha\n(~2013+)", "babyboomers": "Baby Boomers\n(~1946-1964)"}

# ── 1. Revenus totaux par génération ──
rev_gen = df_clean.groupby("age_generation")["salesVatEUR"].agg(
    total_revenue="sum", avg_basket="mean", nb_transactions="count"
).reindex(gen_order).round(2)
print("\n=== REVENUS PAR GÉNÉRATION ===")
print(rev_gen)

# ── 2. Top 10 brands par génération (revenus) ──
brand_gen = df_clean.groupby(["age_generation", "brand"])["salesVatEUR"].sum().reset_index()
brand_gen = brand_gen.sort_values(["age_generation", "salesVatEUR"], ascending=[True, False])
top_brands = brand_gen.groupby("age_generation").head(10)
print("\n=== TOP 10 BRANDS PAR GÉNÉRATION ===")
print(top_brands.to_string())

# ── 3. Top 5 brands par génération — graphique ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Top 5 brands par génération — revenus totaux (€)", fontsize=14, fontweight="bold", y=1.01)

for ax, gen in zip(axes.flatten(), gen_order):
    data = brand_gen[brand_gen["age_generation"] == gen].head(5)
    bars = ax.barh(data["brand"][::-1], data["salesVatEUR"][::-1],
                   color="#534AB7", alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_title(gen_labels[gen], fontsize=11, fontweight="bold")
    ax.set_xlabel("Revenus (€)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k€"))
    ax.grid(axis="x", alpha=0.3)
    for bar, val in zip(bars, data["salesVatEUR"][::-1]):
        ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                f"{val/1000:.1f}k€", va="center", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("brands_par_generation.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

# ── 4. Part de marché par génération (% revenus) ──
brand_share = df_clean.groupby(["age_generation", "brand"])["salesVatEUR"].sum()
brand_share = brand_share / brand_share.groupby("age_generation").transform("sum") * 100
brand_share = brand_share.reset_index()
brand_share.columns = ["age_generation", "brand", "pct_revenus"]
top_share = brand_share.sort_values(["age_generation", "pct_revenus"], ascending=[True, False]).groupby("age_generation").head(5)
print("\n=== PART DE MARCHÉ (%) TOP 5 BRANDS PAR GÉNÉRATION ===")
print(top_share.round(2).to_string())

In [ ]:
# ── Axe produit par génération ──
axe_gen = df_clean.groupby(["age_generation", "Axe_Desc"]).agg(
    total_revenue = ("salesVatEUR", "sum"),
    nb_transactions = ("salesVatEUR", "count"),
    avg_basket = ("salesVatEUR", "mean")
).reset_index().round(2)

print("=== REVENUS PAR AXE ET GÉNÉRATION ===")
print(axe_gen.sort_values(["age_generation","total_revenue"], ascending=[True,False]).to_string())

# ── Transitions brand → brand par génération ──
df_sorted = df_clean.sort_values(["anonymized_card_code", "transactionDate"])
df_sorted["next_brand"] = df_sorted.groupby("anonymized_card_code")["brand"].shift(-1)
df_sorted["next_axe"]   = df_sorted.groupby("anonymized_card_code")["Axe_Desc"].shift(-1)
df_sorted["next_gen"]   = df_sorted.groupby("anonymized_card_code")["age_generation"].shift(-1)

# Garder uniquement les transitions intra-génération (même client)
transitions = df_sorted.dropna(subset=["next_brand", "next_axe"]).copy()

print("\n=== TOP 10 TRANSITIONS BRAND → BRAND PAR GÉNÉRATION ===")
for gen in gen_order:
    t = transitions[transitions["age_generation"] == gen]
    top = t.groupby(["brand","next_brand"]).agg(
        nb = ("salesVatEUR","count"),
        val_next = ("salesVatEUR","mean")
    ).reset_index().sort_values("nb", ascending=False).head(10)
    print(f"\n--- {gen} ---")
    print(top.to_string())

In [ ]:
# ── Corriger MAEK UP dans df_clean ──
df_clean["Axe_Desc"] = df_clean["Axe_Desc"].replace("MAEK UP", "MAKE UP")
print("Vérification :", df_clean["Axe_Desc"].value_counts())

# ── Recalcul revenus par axe et génération ──
axe_gen = df_clean.groupby(["age_generation", "Axe_Desc"]).agg(
    total_revenue   = ("salesVatEUR", "sum"),
    nb_transactions = ("salesVatEUR", "count"),
    avg_basket      = ("salesVatEUR", "mean")
).reset_index().round(2)

print("\n=== REVENUS PAR AXE ET GÉNÉRATION (corrigé) ===")
print(axe_gen.sort_values(["age_generation","total_revenue"], ascending=[True,False]).to_string())

# ── Recalcul transitions avec df_clean corrigé ──
df_sorted = df_clean.sort_values(["anonymized_card_code", "transactionDate"])
df_sorted["next_brand"] = df_sorted.groupby("anonymized_card_code")["brand"].shift(-1)
df_sorted["next_axe"]   = df_sorted.groupby("anonymized_card_code")["Axe_Desc"].shift(-1)
transitions = df_sorted.dropna(subset=["next_brand","next_axe"]).copy()

# ── Top 10 transitions brand → brand par génération ──
print("\n=== TOP 10 TRANSITIONS BRAND → BRAND PAR GÉNÉRATION ===")
for gen in ["genz","geny","gena","babyboomers"]:
    t = transitions[transitions["age_generation"] == gen]
    top = t.groupby(["brand","next_brand"]).agg(
        nb       = ("salesVatEUR", "count"),
        val_next = ("salesVatEUR", "mean")
    ).reset_index().sort_values("nb", ascending=False).head(10)
    print(f"\n--- {gen} ---")
    print(top.round(2).to_string())

# ── Top 5 transitions axe → axe par génération ──
print("\n=== TRANSITIONS AXE → AXE PAR GÉNÉRATION ===")
for gen in ["genz","geny","gena","babyboomers"]:
    t = transitions[transitions["age_generation"] == gen]
    top = t.groupby(["Axe_Desc","next_axe"]).size().reset_index(name="nb")
    top = top.sort_values("nb", ascending=False).head(8)
    print(f"\n--- {gen} ---")
    print(top.to_string())

In [ ]:
# ── Fidélisation : nb de clients avec 2+ achats / nb total ──
client_gen = df_clean.groupby("anonymized_card_code").agg(
    nb_tickets    = ("anonymized_Ticket_ID", "nunique"),
    generation    = ("age_generation", "first"),
    total_spent   = ("salesVatEUR", "sum"),
    nb_brands     = ("brand", "nunique"),
    nb_axes       = ("Axe_Desc", "nunique"),
    first_date    = ("transactionDate", "min"),
    last_date     = ("transactionDate", "max"),
).reset_index()

# Durée de vie client en jours
client_gen["lifespan_days"] = (
    pd.to_datetime(client_gen["last_date"]) - 
    pd.to_datetime(client_gen["first_date"])
).dt.days

# Taux de fidélisation = % clients avec 2+ tickets
fidelisation = client_gen.groupby("generation").apply(lambda x: pd.Series({
    "nb_clients_total"  : len(x),
    "nb_clients_recurrents" : (x["nb_tickets"] >= 2).sum(),
    "taux_fidelisation_pct" : round((x["nb_tickets"] >= 2).mean() * 100, 1),
    "panier_moyen_total"    : round(x["total_spent"].mean(), 2),
    "nb_tickets_moyen"      : round(x["nb_tickets"].mean(), 2),
    "lifespan_moyen_jours"  : round(x["lifespan_days"].mean(), 0),
    "nb_brands_moyen"       : round(x["nb_brands"].mean(), 2),
    "nb_axes_moyen"         : round(x["nb_axes"].mean(), 2),
})).reset_index()

print("=== TAUX DE FIDÉLISATION PAR GÉNÉRATION ===")
print(fidelisation.to_string())

# ── Distribution du nb de tickets par génération ──
print("\n=== DISTRIBUTION NB TICKETS ===")
for gen in ["genz","geny","gena","babyboomers"]:
    d = client_gen[client_gen["generation"] == gen]["nb_tickets"]
    print(f"\n{gen} — médiane: {d.median()}, 75e pct: {d.quantile(0.75)}, max: {d.max()}")
    print(d.value_counts().sort_index().head(8).to_string())

In [ ]:
# ── Profil type par génération : brand favorite + axe favori + canal favori ──
profil = df_clean.groupby("anonymized_card_code").agg(
    generation      = ("age_generation",   "first"),
    brand_favorite  = ("brand",            lambda x: x.mode()[0]),
    axe_favori      = ("Axe_Desc",         lambda x: x.mode()[0]),
    canal_favori    = ("channel",          lambda x: x.mode()[0]),
    store_type      = ("store_type_app",   lambda x: x.mode()[0]),
    nb_tickets      = ("anonymized_Ticket_ID", "nunique"),
    total_spent     = ("salesVatEUR",      "sum"),
    avg_basket      = ("salesVatEUR",      "mean"),
    pct_fragrance   = ("Axe_Desc",         lambda x: (x == "FRAGRANCE").mean() * 100),
    pct_makeup      = ("Axe_Desc",         lambda x: (x == "MAKE UP").mean() * 100),
    pct_skincare    = ("Axe_Desc",         lambda x: (x == "SKINCARE").mean() * 100),
).reset_index()

# ── Agrégation par génération ──
for gen in ["genz", "geny", "gena", "babyboomers"]:
    g = profil[profil["generation"] == gen]
    print(f"\n{'='*50}")
    print(f"PROFIL TYPE — {gen.upper()} ({len(g)} clients)")
    print(f"{'='*50}")
    print(f"Brand favorite      : {g['brand_favorite'].mode()[0]}")
    print(f"Top 3 brands        : {g['brand_favorite'].value_counts().head(3).to_dict()}")
    print(f"Axe favori          : {g['axe_favori'].mode()[0]}")
    print(f"Canal favori        : {g['canal_favori'].mode()[0]}")
    print(f"Store type          : {g['store_type'].mode()[0]}")
    print(f"Tickets moyen       : {g['nb_tickets'].mean():.2f}")
    print(f"Panier moyen        : {g['avg_basket'].mean():.2f}€")
    print(f"Total dépensé moy.  : {g['total_spent'].mean():.2f}€")
    print(f"% achats FRAGRANCE  : {g['pct_fragrance'].mean():.1f}%")
    print(f"% achats MAKE UP    : {g['pct_makeup'].mean():.1f}%")
    print(f"% achats SKINCARE   : {g['pct_skincare'].mean():.1f}%")
    print(f"Canal distribution  :")
    print(g['canal_favori'].value_counts(normalize=True).mul(100).round(1).to_string())
    print(f"Store type dist.    :")
    print(g['store_type'].value_counts(normalize=True).mul(100).round(1).head(4).to_string())

In [ ]:
# ── Dataset de transitions complet ──
df_sorted = df_clean.sort_values(["anonymized_card_code", "transactionDate"])

# Variables du ticket N
# Variables du ticket N+1 (shift -1)
shift_cols = {
    "brand"         : "next_brand",
    "Axe_Desc"      : "next_axe",
    "channel"       : "next_channel",
    "store_type_app": "next_store_type",
    "salesVatEUR"   : "next_basket",
    "Market_Desc"   : "next_market",
}

for col, new_col in shift_cols.items():
    df_sorted[new_col] = df_sorted.groupby("anonymized_card_code")[col].shift(-1)

# Supprimer les derniers achats (pas de N+1)
transitions_full = df_sorted.dropna(subset=["next_brand","next_axe","next_channel"]).copy()

# Vérification
print("Nb transitions :", len(transitions_full))
print("Colonnes disponibles :", transitions_full.shape[1])
print("\nSample :")
print(transitions_full[[
    "anonymized_card_code","gender","age_generation",
    "brand","Axe_Desc","channel",
    "next_brand","next_axe","next_channel","next_basket"
]].head(5).to_string())

In [ ]:
# ── Vérification du genre ──
print("Distribution genre :")
print(df_clean["gender"].value_counts())
print("\nDécoding : 1=homme, 2=femme")

# ── Créer le label genre ──
transitions_full["genre"] = transitions_full["gender"].map({1: "homme", 2: "femme"})
transitions_full = transitions_full[transitions_full["genre"].notna()].copy()

# ── Revenus et comportement par genre ──
print("\n=== REVENUS PAR GENRE ===")
print(df_clean[df_clean["gender"].isin([1,2])].groupby("gender").agg(
    total_revenue   = ("salesVatEUR", "sum"),
    avg_basket      = ("salesVatEUR", "mean"),
    nb_transactions = ("salesVatEUR", "count"),
    nb_clients      = ("anonymized_card_code", "nunique"),
).round(2))

# ── Axe favori par genre ──
print("\n=== AXE PAR GENRE ===")
print(df_clean[df_clean["gender"].isin([1,2])].groupby(["gender","Axe_Desc"])["salesVatEUR"].agg(
    total="sum", avg="mean", nb="count"
).round(2))

# ── Top 5 brands par genre ──
print("\n=== TOP 5 BRANDS PAR GENRE ===")
for g, label in [(1,"homme"),(2,"femme")]:
    sub = df_clean[df_clean["gender"]==g]
    print(f"\n--- {label} ---")
    print(sub.groupby("brand")["salesVatEUR"].sum().sort_values(ascending=False).head(5).round(0))

# ── Canal par genre ──
print("\n=== CANAL PAR GENRE ===")
print(df_clean[df_clean["gender"].isin([1,2])].groupby(["gender","channel"])["salesVatEUR"].count())

# ── Genre × génération × axe — tableau croisé rentabilité ──
print("\n=== GENRE × GÉNÉRATION × AXE (panier moyen) ===")
cross = df_clean[df_clean["gender"].isin([1,2])].groupby(
    ["gender","age_generation","Axe_Desc"]
)["salesVatEUR"].mean().round(2).reset_index()
cross["gender"] = cross["gender"].map({1:"homme",2:"femme"})
print(cross.sort_values(["gender","age_generation","salesVatEUR"], ascending=[True,True,False]).to_string())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# ── Encoder toutes les variables ──
trans = transitions_full.copy()

# Variables profil client
profile_cols = ["gender", "age_generation", "status", "RFM_Segment_ID",
                "days_since_sub", "has_first_purchase_data"]

# Variables achat N (le contexte)
context_cols = ["Axe_Desc", "brand", "channel", "Market_Desc", "salesVatEUR"]

# Tout encoder
le = {}
cat_to_encode = ["age_generation", "Axe_Desc", "brand", "channel",
                 "Market_Desc", "next_axe", "next_brand", "next_channel"]

for col in cat_to_encode:
    le[col] = LabelEncoder()
    trans[col + "_enc"] = le[col].fit_transform(trans[col].astype(str))

# Imputer RFM
trans["RFM_Segment_ID"] = trans["RFM_Segment_ID"].fillna(trans["RFM_Segment_ID"].median())
trans["days_since_sub"] = trans["days_since_sub"].fillna(-1)

# Features finales
features = [
    # Profil client
    "gender", "age_generation_enc", "status", "RFM_Segment_ID",
    "days_since_sub", "has_first_purchase_data",
    # Achat N
    "Axe_Desc_enc", "brand_enc", "channel_enc", "Market_Desc_enc",
    "salesVatEUR",
]

X = trans[features]

# ── Modèle 1 : prédire next_axe ──
y_axe = trans["next_axe_enc"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y_axe, test_size=0.2,
                                            random_state=42, stratify=y_axe)
rf_axe = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 random_state=42, n_jobs=-1)
rf_axe.fit(X_tr, y_tr)
print("── Modèle next_axe ──")
print(classification_report(y_te, rf_axe.predict(X_te),
      target_names=le["next_axe"].classes_))

# ── Modèle 2 : prédire next_channel ──
y_chan = trans["next_channel_enc"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y_chan, test_size=0.2,
                                            random_state=42, stratify=y_chan)
rf_chan = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                  random_state=42, n_jobs=-1)
rf_chan.fit(X_tr, y_tr)
print("\n── Modèle next_channel ──")
print(classification_report(y_te, rf_chan.predict(X_te),
      target_names=le["next_channel"].classes_))

# ── Importance des features ──
import pandas as pd
imp_axe = pd.Series(rf_axe.feature_importances_, index=features).sort_values(ascending=False)
print("\n── Feature importance — next_axe ──")
print(imp_axe.round(4))

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ── Table de règles conditionnelles ──
# Pour chaque combinaison (genre × génération × axe_actuel × canal_actuel)
# → calculer les probabilités de transition vers (next_axe, next_brand, next_channel)

trans["genre"] = trans["gender"].map({1:"homme", 2:"femme"})

# Groupes minimum 30 transitions pour être statistiquement fiables
MIN_TRANSITIONS = 30

rules = []

for (genre, gen, axe, canal), group in trans.groupby(
    ["genre", "age_generation", "Axe_Desc", "channel"]
):
    if len(group) < MIN_TRANSITIONS:
        continue

    # Probabilités next_axe
    next_axe_prob = group["next_axe"].value_counts(normalize=True).head(3)
    # next_brand top 3
    next_brand_top = group["next_brand"].value_counts().head(3)
    # next_channel
    next_chan_prob = group["next_channel"].value_counts(normalize=True)
    # panier moyen suivant
    next_basket = group["next_basket"].mean()

    rules.append({
        "genre"          : genre,
        "generation"     : gen,
        "axe_actuel"     : axe,
        "canal_actuel"   : canal,
        "nb_transitions" : len(group),
        "next_axe_1"     : next_axe_prob.index[0] if len(next_axe_prob) > 0 else None,
        "next_axe_1_pct" : round(next_axe_prob.iloc[0]*100, 1) if len(next_axe_prob) > 0 else None,
        "next_axe_2"     : next_axe_prob.index[1] if len(next_axe_prob) > 1 else None,
        "next_axe_2_pct" : round(next_axe_prob.iloc[1]*100, 1) if len(next_axe_prob) > 1 else None,
        "next_brand_1"   : next_brand_top.index[0] if len(next_brand_top) > 0 else None,
        "next_brand_2"   : next_brand_top.index[1] if len(next_brand_top) > 1 else None,
        "next_brand_3"   : next_brand_top.index[2] if len(next_brand_top) > 2 else None,
        "next_canal_1"   : next_chan_prob.index[0] if len(next_chan_prob) > 0 else None,
        "next_canal_1_pct": round(next_chan_prob.iloc[0]*100, 1) if len(next_chan_prob) > 0 else None,
        "next_basket_moy": round(next_basket, 2),
    })

rules_df = pd.DataFrame(rules).sort_values("nb_transitions", ascending=False)

print(f"Nombre de règles générées : {len(rules_df)}")
print("\n=== TOP 20 RÈGLES LES PLUS FRÉQUENTES ===")
print(rules_df.head(20).to_string())

print("\n=== RÈGLES HOMMES ===")
print(rules_df[rules_df["genre"]=="homme"].to_string())

print("\n=== RÈGLES FRAGRANCE → ??? ===")
print(rules_df[rules_df["axe_actuel"]=="FRAGRANCE"].sort_values("nb_transitions", ascending=False).to_string())

In [ ]:
# ── Règles enrichies avec brand actuelle ──
MIN_TRANSITIONS_BRAND = 20  # seuil plus bas car plus granulaire

rules_brand = []

for (genre, gen, axe, brand, canal), group in trans.groupby(
    ["genre", "age_generation", "Axe_Desc", "brand", "channel"]
):
    if len(group) < MIN_TRANSITIONS_BRAND:
        continue

    next_axe_prob  = group["next_axe"].value_counts(normalize=True).head(2)
    next_brand_top = group["next_brand"].value_counts().head(3)
    next_chan_prob  = group["next_channel"].value_counts(normalize=True)
    next_basket    = group["next_basket"].mean()

    rules_brand.append({
        "genre"          : genre,
        "generation"     : gen,
        "axe_actuel"     : axe,
        "brand_actuelle" : brand,
        "canal_actuel"   : canal,
        "nb_transitions" : len(group),
        "next_axe_1"     : next_axe_prob.index[0] if len(next_axe_prob) > 0 else None,
        "next_axe_1_pct" : round(next_axe_prob.iloc[0]*100,1) if len(next_axe_prob) > 0 else None,
        "next_axe_2"     : next_axe_prob.index[1] if len(next_axe_prob) > 1 else None,
        "next_axe_2_pct" : round(next_axe_prob.iloc[1]*100,1) if len(next_axe_prob) > 1 else None,
        "next_brand_1"   : next_brand_top.index[0] if len(next_brand_top) > 0 else None,
        "next_brand_2"   : next_brand_top.index[1] if len(next_brand_top) > 1 else None,
        "next_brand_3"   : next_brand_top.index[2] if len(next_brand_top) > 2 else None,
        "next_canal_1"   : next_chan_prob.index[0] if len(next_chan_prob) > 0 else None,
        "next_canal_pct" : round(next_chan_prob.iloc[0]*100,1) if len(next_chan_prob) > 0 else None,
        "next_basket_moy": round(next_basket, 2),
    })

rules_brand_df = pd.DataFrame(rules_brand).sort_values("nb_transitions", ascending=False)

print(f"Règles avec brand : {len(rules_brand_df)}")
print("\n=== TOP 20 ===")
print(rules_brand_df.head(20).to_string())
print("\n=== RÈGLES HOMMES avec brand ===")
print(rules_brand_df[rules_brand_df["genre"]=="homme"].sort_values("nb_transitions",ascending=False).head(15).to_string())
print("\n=== RÈGLES FRAGRANCE avec brand ===")
print(rules_brand_df[rules_brand_df["axe_actuel"]=="FRAGRANCE"].sort_values("nb_transitions",ascending=False).head(20).to_string())

In [ ]:
# ── Export table de ciblage finale ──
rules_brand_df["recommandation_pub"] = rules_brand_df.apply(lambda r: (
    f"Pousser {r['next_brand_1']} {r['next_axe_1']} "
    f"via {'app/digital' if r['next_canal_1']=='estore' else 'activation store'}"
), axis=1)

rules_brand_df["valeur_attendue"] = rules_brand_df["next_basket_moy"].apply(
    lambda x: "haute (>50€)" if x > 50 else "moyenne (30-50€)" if x > 30 else "standard (<30€)"
)

# Trier par panier attendu décroissant
output = rules_brand_df[[
    "genre","generation","axe_actuel","brand_actuelle","canal_actuel",
    "nb_transitions",
    "next_axe_1","next_axe_1_pct",
    "next_brand_1","next_brand_2","next_brand_3",
    "next_canal_1","next_canal_pct",
    "next_basket_moy","valeur_attendue","recommandation_pub"
]].sort_values("next_basket_moy", ascending=False)

output.to_csv("sephora_ciblage_894_regles.csv", index=False, encoding="utf-8-sig")
print(f"Fichier exporté : {len(output)} règles")
print("\nTop 10 règles par valeur de panier attendu :")
print(output.head(10)[["genre","generation","axe_actuel","brand_actuelle",
                         "canal_actuel","next_axe_1","next_brand_1",
                         "next_basket_moy","valeur_attendue"]].to_string())

In [ ]:
import pandas as pd

# ── Charger la table de règles ──
rules = pd.read_csv("sephora_ciblage_894_regles.csv")
print("Shape :", rules.shape)
print("Colonnes :", rules.columns.tolist())
print("\nSample :")
print(rules.head(3).to_string())

# ── Vue next_axe × next_brand ──
# Pour chaque (next_axe_1 × next_brand_1) = "ce produit/brand sera acheté"
# On liste tous les profils (genre × generation × axe_actuel × brand_actuelle × canal)
# qui y mènent

target_map = rules.groupby(["next_axe_1","next_brand_1"]).apply(lambda g: pd.Series({
    "nb_profils"       : len(g),
    "profils"          : list(zip(g["genre"], g["generation"], g["axe_actuel"],
                                  g["brand_actuelle"], g["canal_actuel"])),
    "next_basket_moy"  : g["next_basket_moy"].mean().round(2),
    "nb_transitions_total" : g["nb_transitions"].sum(),
    "genres"           : g["genre"].unique().tolist(),
    "generations"      : g["generation"].unique().tolist(),
    "axes_actuels"     : g["axe_actuel"].unique().tolist(),
}), include_groups=False).reset_index()

target_map = target_map.sort_values("nb_profils", ascending=False)

print(f"\nNombre de combinaisons next_axe × next_brand : {len(target_map)}")
print("\n=== TOP 20 PRODUIT/BRAND PAR NB DE PROFILS ===")
print(target_map[["next_axe_1","next_brand_1","nb_profils",
                   "nb_transitions_total","next_basket_moy"]].head(20).to_string())

print("\n=== DISTRIBUTION NB DE PROFILS ===")
print(target_map["nb_profils"].describe())
print("\nValeurs nb_profils :", sorted(target_map["nb_profils"].unique()))

In [ ]:
# ── Correspondance complète : pour chaque next_axe × next_brand → tous les profils ──
rows = []
for _, row in target_map.iterrows():
    for (genre, gen, axe, brand, canal) in row["profils"]:
        # retrouver la règle complète
        rule = rules[
            (rules["next_axe_1"] == row["next_axe_1"]) &
            (rules["next_brand_1"] == row["next_brand_1"]) &
            (rules["genre"] == genre) &
            (rules["generation"] == gen) &
            (rules["axe_actuel"] == axe) &
            (rules["brand_actuelle"] == brand) &
            (rules["canal_actuel"] == canal)
        ].iloc[0]
        rows.append({
            "next_axe"          : row["next_axe_1"],
            "next_brand"        : row["next_brand_1"],
            "nb_profils_total"  : row["nb_profils"],
            "next_basket_moy_group" : row["next_basket_moy"],
            "genre"             : genre,
            "generation"        : gen,
            "axe_actuel"        : axe,
            "brand_actuelle"    : brand,
            "canal_actuel"      : canal,
            "probabilite_pct"   : rule["next_axe_1_pct"],
            "nb_transitions"    : rule["nb_transitions"],
            "next_basket_profil": rule["next_basket_moy"],
            "valeur_attendue"   : rule["valeur_attendue"],
        })

correspondence_df = pd.DataFrame(rows).sort_values(
    ["nb_profils_total", "next_axe", "next_brand", "nb_transitions"],
    ascending=[False, True, True, False]
)

print(f"Lignes totales : {len(correspondence_df)}")
print(f"Combinaisons produit/brand : {correspondence_df[['next_axe','next_brand']].drop_duplicates().shape[0]}")

# Export CSV complet
correspondence_df.to_csv("sephora_correspondance_profils.csv", index=False, encoding="utf-8-sig")
print("\nFichier exporté : sephora_correspondance_profils.csv")

# ── Aperçu lisible pour MAKE UP × SEPHORA (le plus riche) ──
print("\n" + "="*60)
print("MAKE UP × SEPHORA — 434 profils")
print("="*60)
sub = correspondence_df[
    (correspondence_df["next_axe"]=="MAKE UP") &
    (correspondence_df["next_brand"]=="SEPHORA")
][["genre","generation","axe_actuel","brand_actuelle","canal_actuel",
   "probabilite_pct","nb_transitions","next_basket_profil"]].head(20)
print(sub.to_string())

# ── Aperçu FRAGRANCE × DIOR (haute valeur) ──
print("\n" + "="*60)
print("FRAGRANCE × DIOR — 11 profils")
print("="*60)
sub2 = correspondence_df[
    (correspondence_df["next_axe"]=="FRAGRANCE") &
    (correspondence_df["next_brand"]=="DIOR")
][["genre","generation","axe_actuel","brand_actuelle","canal_actuel",
   "probabilite_pct","nb_transitions","next_basket_profil"]]
print(sub2.to_string())